In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

from torch.utils.data import TensorDataset, DataLoader


from tqdm.auto import tqdm

# Load Dataset

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

# Exploratory Data Analyis

## Shape and Sample Check

In [ ]:
print("Dataset Shapes")

print(f"Train Shape : {train.shape}")
print(f"Test Shape  : {test.shape}")
print(f"Submission  : {sample_submission.shape}")

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
sample_submission.head()

In [ ]:
train.info()

In [ ]:
test.info()

## Missing Values and Duplicate Rows

In [ ]:
print("Train Missing Values")

print(train.isnull().sum())

print("\nTrain Duplicate Values")

print(train.duplicated().sum())

## Distribution

In [ ]:
train["answer"].value_counts()

In [ ]:
train["answer"].value_counts().plot(kind="bar",figsize=(6,4))

plt.title("Distribution of Correct Answers")
plt.xlabel("Answer")
plt.ylabel("Count")

plt.show()

## Prompt Length Analysis

In [ ]:
prompt_length= train["prompt"].str.len()

In [ ]:
prompt_length.describe()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(prompt_length, bins=30)

plt.title("Prompt Length Analysis")
plt.xlabel("Characters")
plt.ylabel("Frequency")

plt.show()

### Lets create a copy of training data for functional use

In [ ]:
eda_train = train.copy()

## Option Lengths

In [ ]:
options = ["A", "B", "C", "D", "E"]

for option in options:
  eda_train[f"{option}_length"] = eda_train[option].str.len()

In [ ]:
eda_train[[f"{i}_length" for i in options]].describe()

## Class Balance

In [ ]:
answer_percentage = (eda_train["answer"].value_counts(normalize=True).mul(100).round(2))
display(answer_percentage)

In [ ]:
plt.figure(figsize=(6,4))
answer_percentage.plot(kind="bar")

plt.title("Percentage Distribution of Answer Labels")
plt.xlabel("Answer")
plt.ylabel("Percentage (%)")

plt.xticks(rotation=0)

plt.show()

## Prompt Length by Answer Class


In [ ]:
prompt_analysis = pd.DataFrame({
    "answer": train["answer"],
    "prompt_length": prompt_length
})

prompt_analysis.groupby("answer")["prompt_length"].mean()

## Unique Values

In [ ]:
unique_values = pd.DataFrame({
    "Column": train.columns,
    "Unique Values": [train[col].nunique() for col in train.columns]
})

unique_values

## Memory Usage

In [ ]:
memory_usage = train.memory_usage(deep=True).sum() / (1024**2)

print(f"Memory Usage: {memory_usage:.2f} MB")

### Observations

- The training dataset contains 2,000 samples with 8 Columns and the test dataset contains 500 samples.
- No missing values were found.
- No Duplicate records were detected.
- The answer labels are approximately balanced across all five classes.
- Prompt lengths vary across questions.
- Option lengths are relatively short compared to prompts.
- The dataset occupies only a small amount of memory, making it suitable for experimentation.

In [ ]:
sample=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sample.to_csv("submission.csv", index=False)

# Text Preprocessing

## Inspect One Example

In [ ]:
sample = train.iloc[0]

print(sample["prompt"])
print("\nA :", sample["A"])
print("B :", sample["B"])
print("C :", sample["C"])
print("D :", sample["D"])
print("E :", sample["E"])
print("\nCorrect :", sample["answer"])

## Testing Function

In [ ]:
def clean_text(text):
    text = str(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text

example = "   This      is      a      sample.     "
print(clean_text(example))

## Cleaning

In [ ]:
text_columns = [
    "prompt",
    "A",
    "B",
    "C",
    "D",
    "E"
]

for column in text_columns:
    train[column] = train[column].apply(clean_text)
    test[column] = test[column].apply(clean_text)

# Data Formatting
> Here we convert the original MCQ dataset into a format suitable
for machine learning and transformer models.

## Create MCQ Pairs

In [ ]:
def create_mcq_pairs(df):
    rows = []
    option_columns = ["A", "B", "C", "D", "E"]
    for _, row in df.iterrows():
        for option in option_columns:
            rows.append({
                "id": row["id"],
                "prompt": row["prompt"],
                "option": row[option],
                "option_id": option,
                "label": int(option == row["answer"])
            })
    return pd.DataFrame(rows)

In [ ]:
train_pairs = create_mcq_pairs(train)

#### Convert each MCQ into 5 (prompt, option) pairs.
#### The correct option gets label=1 and the remaining four get label=0.

## Verify the Pairs

In [ ]:
print(train_pairs.shape)
train_pairs.head(10)

# TF-IDF Baseline Model

## Model

In [ ]:
train_pairs["text"] = ("Question: "+ train_pairs["prompt"]+ " Answer: "+ train_pairs["option"])

In [ ]:
train_pairs[["prompt", "option", "text"]].head()

## TF-IDF Vectorizer

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

train_vectors = vectorizer.fit_transform(train_pairs["text"])

print(train_vectors.shape)
print(train_vectors)

## Question Vectorizer

In [ ]:
question_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

# Fit on both questions and options
question_vectorizer.fit(
    pd.concat([
        train_pairs["prompt"],
        train_pairs["option"]
    ])
)

question_vectors = question_vectorizer.transform(train_pairs["prompt"])
option_vectors = question_vectorizer.transform(train_pairs["option"])

## Similarity Score

In [ ]:
similarity_scores = cosine_similarity(
    question_vectors,
    option_vectors
)

In [ ]:
train_pairs["similarity"] = similarity_scores.diagonal()

In [ ]:
train_pairs[["prompt", "option", "similarity"]].head(10)

## Ranking

In [ ]:
train_pairs["rank"] = (
    train_pairs
    .groupby("id")["similarity"]
    .rank(method="first", ascending=False)
)

## Top 3 Predictions

In [ ]:
top3 = (
    train_pairs
    .sort_values(["id", "similarity"], ascending=[True, False])
    .groupby("id")
    .head(3)
)

top3.head()

In [ ]:
top3_predictions = (
    train_pairs
    .sort_values(["id", "similarity"], ascending=[True, False])
    .groupby("id")["option_id"]
    .apply(list)
    .apply(lambda x: x[:3])
)

top3_predictions.head()

## Ground Truth

In [ ]:
ground_truth = (
    train_pairs[train_pairs["label"] == 1]
    .set_index("id")["option_id"]
)

ground_truth.head()

## MAP@3 Function

In [ ]:
def map_at_3(actual, predicted):
    score = 0.0

    for qid in actual.index:

        true_answer = actual[qid]
        predictions = predicted[qid]

        if true_answer in predictions:
            rank = predictions.index(true_answer) + 1
            score += 1 / rank

    return score / len(actual)

In [ ]:
score = map_at_3(ground_truth, top3_predictions)

print(f"MAP@3 Score: {score:.4f}")

# Transformer

## Train/Validation Split

In [ ]:
train_questions, valid_questions = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["answer"]
)

train_df = create_mcq_pairs(train_questions)
valid_df = create_mcq_pairs(valid_questions)

## Load Tokenizer

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    force_download=True
)

## Creating Hugging Face Dataset

In [ ]:
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

In [ ]:
train_dataset[0]

## Tokenizer

In [ ]:
def tokenize(example):
    return tokenizer(
        example["prompt"],
        example["option"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

In [ ]:
train_dataset = train_dataset.map(tokenize, batched=True)
valid_dataset = valid_dataset.map(tokenize, batched=True)

In [ ]:
if "label" in train_dataset.column_names:
    train_dataset = train_dataset.rename_column("label", "labels")

if "label" in valid_dataset.column_names:
    valid_dataset = valid_dataset.rename_column("label", "labels")

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

valid_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "token_type_ids",
        "labels"
    ]
)

## Load The Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

## Metric

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="binary")
    }

## Training Arguments

In [ ]:
BERT_OUTPUT = "/kaggle/working/bert_output"

training_args = TrainingArguments(
    output_dir= BERT_OUTPUT,

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=100,

    report_to="none"
)

## Trainer

In [ ]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
print("\nValidation MAP@3 (BERT)")

bert_encodings = tokenizer(
    valid_df["prompt"].tolist(),
    valid_df["option"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

bert_loader = DataLoader(
    TensorDataset(
        bert_encodings["input_ids"],
        bert_encodings["attention_mask"],
        bert_encodings["token_type_ids"]
    ),
    batch_size=32,
    shuffle=False
)

bert_scores = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

with torch.no_grad():

    for batch in bert_loader:

        batch = [x.to(device) for x in batch]

        outputs = model(
            input_ids=batch[0],
            attention_mask=batch[1],
            token_type_ids=batch[2]
        )

        probs = torch.softmax(outputs.logits, dim=1)

        bert_scores.extend(probs[:,1].cpu().numpy())

In [ ]:
bert_results = valid_df.copy()

bert_results["score"] = bert_scores

In [ ]:
bert_top3 = (
    bert_results
    .sort_values(["id","score"],ascending=[True,False])
    .groupby("id")
    .head(3)
)

bert_predictions = (
    bert_top3
    .groupby("id")["option_id"]
    .apply(list)
    .to_dict()
)

ground_truth = (
    bert_results[bert_results.label==1]
    .set_index("id")["option_id"]
    .to_dict()
)

In [ ]:
def map3(actual,pred):

    for i,p in enumerate(pred[:3]):

        if p==actual:

            return 1/(i+1)

    return 0

In [ ]:
bert_map3=np.mean([
    map3(ground_truth[q],bert_predictions[q])
    for q in ground_truth
])

print("BERT MAP@3 =",round(bert_map3,4))

In [ ]:
trainer.save_model(BERT_OUTPUT)
tokenizer.save_pretrained(BERT_OUTPUT)

print(f"BERT Model saved to: {BERT_OUTPUT}")

# LoRA Fine-Tuning

## LoRA Configuration

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "key", "value"],
    modules_to_save=["classifier"]
)

## LoRA Fresh BERT 

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

## Apply Lora

In [ ]:
model = get_peft_model(model,lora_config)

model.print_trainable_parameters()

## Training Arguments

In [ ]:
LORA_OUTPUT = "/kaggle/working/lora_output"

lora_training_args = TrainingArguments(
    output_dir = LORA_OUTPUT,

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-4,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=8,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_steps=100,

    report_to="none"
)

## Trainer

In [ ]:
lora_trainer = Trainer(
    model=model,

    args=lora_training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    compute_metrics=compute_metrics
)

In [ ]:
lora_trainer.train()

## Evaluate

In [ ]:
lora_results = lora_trainer.evaluate()

print(lora_results)

In [ ]:
print("\nValidation MAP@3 (LoRA)")

lora_encodings = tokenizer(
    valid_df["prompt"].tolist(),
    valid_df["option"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

lora_loader = DataLoader(
    TensorDataset(
        lora_encodings["input_ids"],
        lora_encodings["attention_mask"],
        lora_encodings["token_type_ids"]
    ),
    batch_size=32,
    shuffle=False
)

lora_scores = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

model.eval()

with torch.no_grad():

    for batch in lora_loader:

        batch = [x.to(device) for x in batch]

        outputs = model(
            input_ids=batch[0],
            attention_mask=batch[1],
            token_type_ids=batch[2]
        )

        probs = torch.softmax(outputs.logits, dim=1)

        lora_scores.extend(probs[:,1].cpu().numpy())

In [ ]:
lora_results = valid_df.copy()

lora_results["score"] = lora_scores

In [ ]:
lora_top3 = (
    lora_results
    .sort_values(["id","score"],ascending=[True,False])
    .groupby("id")
    .head(3)
)

lora_predictions = (
    lora_top3
    .groupby("id")["option_id"]
    .apply(list)
    .to_dict()
)

ground_truth = (
    lora_results[bert_results.label==1]
    .set_index("id")["option_id"]
    .to_dict()
)

In [ ]:
def map3(actual,pred):

    for i,p in enumerate(pred[:3]):

        if p==actual:

            return 1/(i+1)

    return 0

In [ ]:
lora_map3=np.mean([
    map3(ground_truth[q],lora_predictions[q])
    for q in ground_truth
])

print("LoRA MAP@3 =",round(lora_map3,4))

In [ ]:
lora_trainer.save_model(LORA_OUTPUT)

tokenizer.save_pretrained(LORA_OUTPUT)

print(f"LoRA model saved to: {LORA_OUTPUT}")

## Model Comparison

In [ ]:
comparison=pd.DataFrame({
    "Model":[
        "BERT",
        "LoRA"
    ],
    "MAP@3":[
        bert_map3,
        lora_map3
    ]
})

comparison

# Inference & Submission


## Best Model

In [ ]:
print("\nModel Performance")
print("-----------------------")
print(f"BERT   MAP@3 : {bert_map3:.4f}")
print(f"LoRA   MAP@3 : {lora_map3:.4f}")

best_model = max(
    {
        "bert": bert_map3,
        "lora": lora_map3
    },
    key={
        "bert": bert_map3,
        "lora": lora_map3
    }.get
)

print(f"\nBest Model: {best_model}")

## Load Model

In [ ]:
# ==========================================================
# Load Best Model
# ==========================================================

if best_model == "bert":

    MODEL_PATH = "/kaggle/working/bert_output"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH
    )

elif best_model == "lora":

    MODEL_PATH = "/kaggle/working/lora_output"

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    from peft import PeftModel

    base_model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2
    )

    model = PeftModel.from_pretrained(
        base_model,
        MODEL_PATH
    )

else:
    raise ValueError(
        "TF-IDF is not used for Transformer inference."
    )

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

model.eval()

print(f"\nUsing Model : {best_model}")
print(f"Using Device: {device}")

## Test to Prompt

In [ ]:
# Create Prompt-Option Pairs for Test Set

test_pairs = []

for _, row in test.iterrows():

    for option in ["A", "B", "C", "D", "E"]:

        test_pairs.append({
            "id": row["id"],
            "prompt": row["prompt"],
            "option": row[option],
            "option_id": option
        })

test_pairs = pd.DataFrame(test_pairs)

print(test_pairs.shape)

test_pairs.head()

## Tokenize Test Data 

In [ ]:
test_encodings = tokenizer(
    test_pairs["prompt"].tolist(),
    test_pairs["option"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=256,
    return_tensors="pt"
)

## Move Data to GPU

In [ ]:
test_encodings = {
    key: value.to(device)
    for key, value in test_encodings.items()
}

## Generate Predictions

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Create Dataset
if "token_type_ids" in test_encodings:
    test_dataset = TensorDataset(
        test_encodings["input_ids"].cpu(),
        test_encodings["attention_mask"].cpu(),
        test_encodings["token_type_ids"].cpu()
    )
else:
    test_dataset = TensorDataset(
        test_encodings["input_ids"].cpu(),
        test_encodings["attention_mask"].cpu()
    )

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

scores = []

model.eval()

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)

        if len(batch) == 3:
            token_type_ids = batch[2].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        else:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        probs = torch.softmax(outputs.logits, dim=1)

        scores.extend(probs[:, 1].cpu().numpy())

## Attach Scores

In [ ]:
test_pairs["score"] = scores

test_pairs.head()

## Select Top 3 Options

In [ ]:
# Select Top 3 Predictions

submission = (
    test_pairs
    .sort_values(["id", "score"], ascending=[True, False])
    .groupby("id")
    .head(3)
)

In [ ]:
submission = (
    submission
    .groupby("id")["option_id"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

submission.columns = ["id", "Prediction"]

submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)

print("Submission file saved successfully!")

print(submission.shape)

assert submission.shape[0] == test["id"].nunique()

assert submission["Prediction"].str.split().str.len().eq(3).all()

submission.head()